In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# LOAD DATASET

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load
fake = pd.read_csv('/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv')
true = pd.read_csv('/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv')

# Add labels
fake['label'] = 0  # 0 = Fake
true['label'] = 1  # 1 = True

# Combine
df = pd.concat([fake, true], ignore_index=True)

print("Total articles:", len(df))
print("\nLabel distribution:\n", df['label'].value_counts())
print("\nColumns:", df.columns.tolist())
print("\nSample fake text (first 200 chars):\n", df[df['label']==0]['text'].iloc[0][:200])

# PREPROCESSING ( REMOVING DUPLICATE )

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)         # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)

# Count duplicates
duplicates = df.duplicated(subset='clean_text').sum()
print(f"Duplicate articles (exact text): {duplicates}")

# Remove duplicates (keep first occurrence)
df_clean = df.drop_duplicates(subset='clean_text', keep='first').reset_index(drop=True)
print(f"After deduplication: {len(df_clean)} articles")

# Label distribution after cleaning
print("\nLabel distribution after cleaning:")
print(df_clean['label'].value_counts())

# Save cleaned dataset for later steps
df_clean.to_csv('/kaggle/working/clean_data.csv', index=False)
print("\nSaved clean_data.csv")

# FIGURES

**1. Class distribution before and after deduplication**

In [ ]:
#Figure 1: Class distribution before and after deduplication

import pandas as pd
import matplotlib.pyplot as plt

# Load cleaned data (from Step 2)
df_clean = pd.read_csv('/kaggle/working/clean_data.csv')

# Original counts (from Step 1 – we know them)
original_fake = 23481
original_true = 21417

# Clean counts
clean_counts = df_clean['label'].value_counts().sort_index()
clean_fake = clean_counts[0]
clean_true = clean_counts[1]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before cleaning
axes[0].bar(['Fake', 'True'], [original_fake, original_true], color=['red', 'green'])
axes[0].set_title('Before Cleaning (44,898 articles)')
axes[0].set_ylabel('Count')
for i, v in enumerate([original_fake, original_true]):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# After cleaning
axes[1].bar(['Fake', 'True'], [clean_fake, clean_true], color=['red', 'green'])
axes[1].set_title(f'After Cleaning ({len(df_clean)} articles)')
axes[1].set_ylabel('Count')
for i, v in enumerate([clean_fake, clean_true]):
    axes[1].text(i, v + 200, str(v), ha='center', fontweight='bold')

plt.suptitle('Data Leakage: Duplicate Articles Removed', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/fig1_class_distribution.png', dpi=150)
plt.show()
print("Saved: fig1_class_distribution.png")

**2. Article length distribution (Fake vs True)**

In [ ]:
#Figure 2: Article length distribution (Fake vs True)

# Ensure clean_text is string
df_clean['clean_text'] = df_clean['clean_text'].astype(str)

# Add text length column
df_clean['text_length'] = df_clean['clean_text'].apply(len)

plt.figure(figsize=(10, 5))
plt.hist(df_clean[df_clean['label']==0]['text_length'], bins=50, alpha=0.6, color='red', label='Fake')
plt.hist(df_clean[df_clean['label']==1]['text_length'], bins=50, alpha=0.6, color='green', label='True')
plt.xlabel('Article Length (characters)')
plt.ylabel('Frequency')
plt.title('Article Length Distribution: Fake vs True News')
plt.legend()
plt.xlim(0, 20000)
plt.tight_layout()
plt.savefig('/kaggle/working/fig2_length_distribution.png', dpi=150)
plt.show()
print("Saved: fig2_length_distribution.png")

**3. Duplicate analysis (bar chart)**

In [ ]:
#Figure 3: Duplicate analysis (bar chart for paper)

fig, ax = plt.subplots(figsize=(8, 5))
categories = ['Original\nDataset', 'Duplicate\nArticles', 'Clean\nDataset']
values = [44898, 6313, len(df_clean)]
colors = ['blue', 'orange', 'green']
bars = ax.bar(categories, values, color=colors, width=0.5)
ax.set_ylabel('Number of Articles')
ax.set_title('Data Leakage Analysis: Duplicate Detection')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200, f'{val:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/fig3_leakage_analysis.png', dpi=150)
plt.show()
print("Saved: fig3_leakage_analysis.png")

**4. Meta‑features distributions (caps ratio, exclamations, avg word length)**

In [ ]:
#Figure 4: Meta‑features distributions (caps ratio, exclamations, avg word length)

import numpy as np

# Ensure clean_text is string (again, just in case)
df_clean['clean_text'] = df_clean['clean_text'].astype(str)

# Compute extra meta features
df_clean['caps_ratio'] = df_clean['clean_text'].apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1))
df_clean['exclamation_count'] = df_clean['clean_text'].apply(lambda x: x.count('!'))
df_clean['avg_word_length'] = df_clean['clean_text'].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
features = ['text_length', 'exclamation_count', 'avg_word_length']
titles = ['Article Length', 'Exclamation Marks', 'Avg Word Length']
for ax, feat, title in zip(axes, features, titles):
    df_clean[df_clean['label']==0][feat].hist(bins=40, alpha=0.6, color='red', label='Fake', ax=ax)
    df_clean[df_clean['label']==1][feat].hist(bins=40, alpha=0.6, color='green', label='True', ax=ax)
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(left=0)
plt.suptitle('Meta-Feature Distributions: Fake vs True News', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/fig4_meta_features.png', dpi=150)
plt.show()
print("Saved: fig4_meta_features.png")